# app_1_api_merger Combined
This notebook combines all scripts from `app_1_api_merger`.

### api_data_collection.py

In [ ]:
import pandas as pd
import requests
import time
from pytrends.request import TrendReq
from datetime import datetime
import os
import random
import sys

# Windows Unicode fix
if sys.platform == 'win32':
    sys.stdout.reconfigure(encoding='utf-8')

# Use Pageviews API or just standard requests
def get_wikipedia_pageviews(article, start_date="20200101", end_date="20241231"):
    if pd.isna(article) or article == "NULL":
        return pd.DataFrame()
        
    article = article.replace(" ", "_")
    url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/user/{article}/monthly/{start_date}/{end_date}"
    
    headers = {
        'User-Agent': 'TourismResilienceResearchBot/1.0 (contact@research.edu)'
    }
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            data = response.json()
            if 'items' in data:
                df = pd.DataFrame(data['items'])
                df['month'] = pd.to_datetime(df['timestamp'], format='%Y%m%d%H').dt.to_period('M')
                return df[['month', 'views']]
    except Exception as e:
        print(f"Error fetching wiki for {article}: {e}")
        
    return pd.DataFrame()

def get_google_trends(keyword, start_year=2020, end_year=2024):
    if pd.isna(keyword) or keyword == "NULL" or len(keyword) < 2:
        return pd.DataFrame()
        
    pytrends = TrendReq(hl='en-US', tz=360)
    # We fetch for an extended period to get monthly data.
    timeframe = f"{start_year}-01-01 {end_year}-12-31"
    
    try:
        pytrends.build_payload([keyword], cat=0, timeframe=timeframe, geo='', gprop='')
        df = pytrends.interest_over_time()
        time.sleep(random.uniform(2, 4)) # sleep to prevent rate limiting
        
        if not df.empty and keyword in df.columns:
            df = df.reset_index()
            df['month'] = df['date'].dt.to_period('M')
            # aggregate to monthly if returned as weekly
            monthly_df = df.groupby('month')[keyword].sum().reset_index()
            monthly_df.rename(columns={keyword: 'trends_index'}, inplace=True)
            return monthly_df
    except Exception as e:
        print(f"Error fetching trends for {keyword}: {e}")
        time.sleep(10) # Heavy sleep if backed off
        
    return pd.DataFrame()

def collect_data_for_destinations(input_csv, overwrite=False):
    # Read regions and attractions
    df = pd.read_csv(input_csv)
    
    # We will accumulate data for all regions
    all_metrics = []
    
    for idx, row in df.iterrows():
        region = row['region']
        country = row['country']
        capital = row['capital']
        attractions = [
            row.get('tourist_destination_1', 'NULL'),
            row.get('tourist_destination_2', 'NULL'),
            row.get('tourist_destination_3', 'NULL')
        ]
        
        print(f"Collecting API data for Region: {region}")
        
        # Collect wiki for region, capital and country
        wiki_terms = [region, capital, country] + attractions
        wiki_terms = list(set([t for t in wiki_terms if pd.notna(t) and t != "NULL"]))
        
        region_df = pd.DataFrame()
        
        # Accumulate metrics per month
        for term in wiki_terms:
            wiki_res = get_wikipedia_pageviews(term)
            if not wiki_res.empty:
                wiki_res.rename(columns={'views': f'wiki_views_{term}'}, inplace=True)
                if region_df.empty:
                    region_df = wiki_res
                else:
                    region_df = pd.merge(region_df, wiki_res, on='month', how='outer')
                    
            # Uncomment below to enable Google Trends. Note: PyTrends gets aggressive rate limits. 
            # trends_res = get_google_trends(term)
            # if not trends_res.empty:
            #     trends_res.rename(columns={'trends_index': f'gt_index_{term}'}, inplace=True)
            #     if region_df.empty:
            #         region_df = trends_res
            #     else:
            #         region_df = pd.merge(region_df, trends_res, on='month', how='outer')
        
        if not region_df.empty:
            region_df['region'] = region
            region_df['country'] = country
            # Sum up total views across the region, capital and attractions
            wiki_cols = [c for c in region_df.columns if c.startswith('wiki_views_')]
            region_df['total_wiki_views'] = region_df[wiki_cols].sum(axis=1)
            all_metrics.append(region_df)
    
    if all_metrics:
        final_df = pd.concat(all_metrics, ignore_index=True)
        # Final formatting
        final_df['month'] = final_df['month'].astype(str)
        out_path = 'app_1_api_merger/api_collected_data.csv'
        final_df.to_csv(out_path, index=False)
        print(f"Data collection complete! Saved to {out_path}")
    else:
        print("No data collected.")

if __name__ == "__main__":
    # We will point it to the local user dataset path
    # On Windows execution it's accessed via WSL path
    dataset_path = r"\\wsl.localhost\Ubuntu\home\fred\viral-tourism-resilience\data_setup\00_Raw_Data\regions_capitals_countries_tourism.csv"
    if os.path.exists(dataset_path):
        collect_data_for_destinations(dataset_path)
    else:
        print(f"Could not find dataset at {dataset_path}")


### data_overview_analysis.py

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

def overview_analysis(dataset_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    
    # Load dataset
    print(f"Loading {dataset_path} for an overview analysis...")
    df = pd.read_excel(dataset_path, sheet_name=0)
    
    # Check what columns exist. We assume it's the Eurostat dataset.
    print(f"Dataset Shape: {df.shape}")
    print(f"Dataset Columns: {df.columns.tolist()}")

    # Overview info
    summary_path = os.path.join(out_dir, "dataset_overview.txt")
    with open(summary_path, "w") as f:
        f.write("=== Dataset Overview ===\n")
        f.write(f"Shape: {df.shape}\n")
        f.write(f"Columns: {df.columns.tolist()}\n")
        
        # Missing values
        f.write("\n=== Missing Values ===\n")
        f.write(df.isnull().sum().to_string())
        
        # Describe
        f.write("\n\n=== Descriptive Statistics ===\n")
        f.write(df.describe().to_string())
        
    print(f"Summary written to {summary_path}")

    # Identify numeric columns for correlation heatmap
    numeric_df = df.select_dtypes(include=['float64', 'int64'])
    
    if not numeric_df.empty:
        plt.figure(figsize=(10, 8))
        sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
        plt.title('Correlation Heatmap of Numeric Attributes')
        plt.tight_layout()
        heatmap_path = os.path.join(out_dir, "correlation_heatmap.png")
        plt.savefig(heatmap_path)
        plt.close()
        print(f"Heatmap saved to {heatmap_path}")
    else:
        print("No numeric data available for heatmap.")

if __name__ == "__main__":
    base_data = r"\\wsl.localhost\Ubuntu\home\fred\viral-tourism-resilience\data_setup\00_Raw_Data\tour_occ_nin2m$defaultview_spreadsheet.xlsx"
    if os.path.exists(base_data):
        overview_analysis(base_data, "app_1_api_merger/analysis_results")
    else:
        print("Base dataset not found.")


### merge_destinations.py

In [ ]:
import pandas as pd
import os
import glob
from openpyxl import load_workbook

def merge_data(eurostat_dir, api_path, out_path):
    print("Loading datasets...")
    
    # 1. Load our collected API data
    if not os.path.exists(api_path):
        print(f"No API data found at {api_path}. Run api_data_collection.py first.")
        return

    api_df = pd.read_csv(api_path)
    # the api data has columns like: month, wiki_views_..., country, region, total_wiki_views
    
    # Aggregate API data to yearly/total level since Eurostat wide format is complex to map month-by-month immediately
    api_agg = api_df.groupby('region')['total_wiki_views'].sum().reset_index()

    # 2. Find and load Eurostat
    f_list = glob.glob(os.path.join(eurostat_dir, '*.xlsx'))
    if not f_list:
        print("Eurostat Excel file not found!")
        return
        
    eurostat_path = f_list[0]
    try:
        # Just process Sheet 1 (assuming 2020) for demonstration of pipeline
        raw = pd.read_excel(eurostat_path, sheet_name='Sheet 1', header=None, dtype=str)
        raw = raw.fillna('')
        
        geo_row_idx = None
        for i, r in raw.iterrows():
            if str(r.iloc[0]).strip() == "GEO (Labels)":
                geo_row_idx = i
                break
                
        if geo_row_idx is not None:
            # We treat the remaining rows as regions. The columns 1, 3, 5 etc represent Jan, Feb, Mar...
            euro_df = raw.iloc[geo_row_idx + 1:].copy()
            euro_df.columns = [f"col_{i}" for i in range(len(euro_df.columns))]
            euro_df['region'] = euro_df['col_0'].str.strip()
            
            # Combine all numeric month columns to get a 'total_visitors' approximation
            # Note: the Eurostat data has numbers in 'col_1', 'col_3' etc. separated by empty cols or ':' for missing
            def sum_visitors(row):
                total = 0
                for c in row.index:
                    if c != 'region' and c != 'col_0':
                        val = str(row[c]).replace(':', '').strip()
                        if val.isdigit():
                            total += int(val)
                return total
                
            euro_df['total_visitors'] = euro_df.apply(sum_visitors, axis=1)
            euro_agg = euro_df[['region', 'total_visitors']]
            
            # 3. Merge!
            merged_df = pd.merge(euro_agg, api_agg, on='region', how='inner')
            merged_df.to_csv(out_path, index=False)
            print(f"Data successfully merged based on extracted visitors! Final dataset saved to {out_path}")
        else:
            print("Could not parse Eurostat format structure (GEO Labels missing).")
            
    except Exception as e:
        print(f"Error during merge: {e}")

if __name__ == "__main__":
    euro_dir = r"\\wsl.localhost\Ubuntu\home\fred\viral-tourism-resilience\data_setup\00_Raw_Data"
    api_file = "app_1_api_merger/api_collected_data.csv"
    output_dest = "app_1_api_merger/all_destinations.csv"
    
    merge_data(euro_dir, api_file, output_dest)
